# P15 – Digitalni filtar na zvuku: pomični prosjek i IIR filtar prvog reda

Bilježnica uz predavanje P15 i **poglavlja 14 i 15 udžbenika *Signali i sustavi***
(z-transformacija; prijenosna funkcija diskretnog sustava). Zapis sa šumom i piskom propuštamo
kroz dva niskopropusna digitalna filtra: **pomični prosjek** duljine $M$ (FIR) i **IIR filtar
prvog reda** dobiven iz RC kruga s vremenskom konstantom $\tau_f$ trima metodama
diskretizacije: Eulerovom, zadržavanjem nultog reda (ZOH) i Tustinovom. Za svaki filtar crtamo
nule i polove u $z$-ravnini i modul frekvencijskog odziva $|H(e^{j\varOmega})|$ u hercima, a
izlaz slušamo i uspoređujemo s ulazom. Klizačima se biraju granična frekvencija $f_c$
(frekvencija loma RC kruga, pogl. 11) i duljina prosjeka $M$. Zadani zapis je sintetički
(melodija, šum i pisak); umjesto njega možete učitati vlastiti zapis (WAV), npr. govor uz
šum ventilatora.

**Kako pokrenuti.** Ćelije pokrenite redom (Shift + Enter) ili izbornikom *Run → Run All
Cells*. Instalaciju Pythona i JupyterLaba opisuje `README.md` repozitorija. Bilježnica radi bez
interneta, a zadani zapis sama sintetizira. Za slušanje trebaju zvučnici ili slušalice; bez njih
rade grafovi i ispis. Ako se klizači ne prikažu, funkciju pokusa pozovite izravno (odjeljak 4).

In [ ]:
import html
import os
import warnings
from math import gcd
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.io import wavfile
from scipy.signal import resample_poly
from IPython.display import Audio, display
import ipywidgets as widgets

MOJ_ZVUK = Path("moj_zvuk")            # vlastiti zapisi (nije u gitu)
BOJA = "#840B55"                       # bordo: Tustinov model i izlaz filtra
SIVA, TAMNA = "#5B6770", "#222222"
# Krivulje se razlikuju i vrstom crte ili oznakom, pa su čitljive i crno-bijelo.
STIL = {"pomični prosjek": dict(color=TAMNA, ls="-", lw=1.4),
        "Euler": dict(color=SIVA, ls="--", lw=1.6),
        "ZOH": dict(color=SIVA, ls="-.", lw=1.6),
        "Tustin": dict(color=BOJA, ls=":", lw=2.2)}

# decimalni zarez i pravi minus na osima
zarez = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})

## Postavke: zadani ili vlastiti zapis

Bilježnica zadano filtrira sintetički zapis koji sama izračuna (odjeljak 2). Za **vlastiti
zapis** upišite njegovu putanju između navodnika u ćeliji ispod (`SNIMKA`), pa pokrenite sve
ćelije (*Run → Run All Cells*). Prazni navodnici `""` znače zadani zapis. Druga mogućnost je gumb
za prijenos datoteke u idućoj ćeliji.

Bilježnica čita samo **WAV** (mono ili stereo, 8, 16, 24 ili 32 bita, bilo koja frekvencija
uzorkovanja). Snimku iz mobitela (M4A, MP3, OGG, FLAC …) najprije pretvorite u WAV, npr.
besplatnim programom **Audacity**: *File → Open* (otvorite snimku), zatim *File → Export
Audio…* (u starijim inačicama *File → Export → Export as WAV*), vrsta datoteke *WAV*,
kodiranje *Signed 16-bit PCM*. Ako je instaliran program ffmpeg, isto radi naredba
`ffmpeg -i snimka.m4a snimka.wav`.

Ostale postavke: frekvencija uzorkovanja `FS` (vlastiti zapis se na nju preuzorkuje; period
uzorkovanja je $T_s = 1/f_s$), sastav zadanog zapisa i mreže klizača. Na kraju su brojke iz
knjige s kojima bilježnica uspoređuje svoje funkcije (odjeljak 3).

In [ ]:
# ===== POSTAVKE (mijenjajte po želji, pa Run → Run All Cells) =====
# Vlastiti zapis: putanja relativna prema mapi bilježnice. Prazno ("") = zadani sintetički.
SNIMKA = ""              # npr. "moj_zvuk/govor_ventilator.wav"
FS = 16000               # Hz: frekvencija uzorkovanja f_s (vlastiti zapis se preuzorkuje)
MAX_SNIMKA_S = 10        # s: dulji vlastiti zapis skraćuje se

# Zadani sintetički zapis: melodija tonova s harmonicima, šum (šuštanje) i stalni pisak
MELODIJA = (262, 294, 330, 349, 392, 349, 330, 294, 262, 330)   # Hz: osnovni tonovi redom
TRAJANJE_TONA = 0.4      # s
HARMONICI = (0.3, 0.15, 0.08, 0.04)   # amplitude osnovnog tona i triju viših harmonika
SUM = 0.04               # standardna devijacija šuma
F_PISAK = 2000           # Hz: frekvencija piska (npr. zavojnica punjača)
C_PISAK = 0.04           # amplituda piska
SJEME = 15               # sjeme generatora slučajnih brojeva (šum)

# Klizači
F_C_IZBOR = (300, 400, 500, 700, 1000, 1400, 2000, 2800, 4000)   # Hz: granična frekvencija RC kruga
F_C_POCETNA = 1000
M_NAJVECI = 24           # najveća duljina pomičnog prosjeka (klizač M od 1)
M_POCETNI = 4
FILTRI = ("pomični prosjek", "Euler", "ZOH", "Tustin")
FILTAR_POCETNI = "Tustin"

# Iz knjige (poglavlje 15, odjeljak 3): primjer pomičnog prosjeka i RC krug iz primjera
# Eulerove metode, ZOH i Tustinove metode
KNJIGA_FIR_FS = 500      # Hz: frekvencija uzorkovanja mikrokontrolera
KNJIGA_FIR_M = 10        # duljina prosjeka
KNJIGA_FIR_F = 75        # Hz: smetnja između nula
KNJIGA_FIR_H = 0.220     # |H| na KNJIGA_FIR_F
KNJIGA_RC_TAU = 0.5      # s: vremenska konstanta RC kruga iz primjera
KNJIGA_RC_TS = 0.1       # s: period uzorkovanja u primjeru
KNJIGA_POL_EULER = 0.8
KNJIGA_POL_ZOH = 0.8187
KNJIGA_POL_TUSTIN = 0.8182
KNJIGA_TUSTIN_B = 0.0909  # koeficijent Tustinova modela uz (z + 1)

# Za automatske testove: prazno polje preuzima vrijednost varijable okoline P15_SNIMKA, ako
# je postavljena. Na predavanju to ne treba: upišite putanju gore.
SNIMKA = SNIMKA or os.environ.get("P15_SNIMKA", "")

In [ ]:
# Prijenos snimke gumbom (JupyterLab; radi i bez interneta). Gumb otvori odabir datoteke.
# Prenesena datoteka sprema se u moj_zvuk/ i zamjenjuje SNIMKA iz ćelije iznad. Nakon
# prijenosa ponovno pokrenite sve ćelije (Run → Run All Cells): gumb pamti datoteku do
# ponovnog pokretanja jezgre (Kernel → Restart Kernel) ili do naredbe gumb_snimka.value = ().
# Bez prijenosa ćelija ništa ne čeka. Datoteke veće od oko 10 MB radije kopirajte u moj_zvuk/
# i upišite putanju gore.


def spremi_prijenos(preneseno, mapa=MOJ_ZVUK):
    # zadnja datoteka prenesena gumbom -> spremljena u mapu; vraća njezinu putanju ili ""
    if not preneseno:
        return ""
    datoteka = preneseno[-1]                   # ipywidgets 8: n-torka rječnika
    mapa = Path(mapa)
    mapa.mkdir(parents=True, exist_ok=True)
    put = mapa / Path(datoteka["name"]).name    # samo ime, bez mape iz preglednika
    put.write_bytes(bytes(datoteka["content"]))
    return put.as_posix()


if "gumb_snimka" not in globals():              # isti gumb i nakon Run All Cells
    gumb_snimka = widgets.FileUpload(accept=".wav", description="snimka (WAV)")
    stanje = widgets.HTML("nije prenesena nijedna datoteka")

    def _preneseno(_promjena):
        if gumb_snimka.value:
            ime = html.escape(gumb_snimka.value[-1]["name"])
            stanje.value = f"preneseno: {ime}; sada <b>Run → Run All Cells</b>"

    gumb_snimka.observe(_preneseno, "value")

display(widgets.HBox([gumb_snimka, stanje]))
SNIMKA = spremi_prijenos(gumb_snimka.value) or SNIMKA

### Kako snimiti vlastiti zapis

1. Snimite nekoliko sekundi govora ili glazbe uz stalni šum: ventilator, usisavač, šuštanje
   radija između postaja ili zujanje računala. Mobitel držite tridesetak centimetara od izvora.
2. Snimku pretvorite u WAV (vidi gore) i spremite je u mapu `moj_zvuk/`, npr. kao
   `moj_zvuk/govor_ventilator.wav`. U postavkama upišite
   `SNIMKA = "moj_zvuk/govor_ventilator.wav"`.

Bilježnica snimku pripremi sama i ispiše svaku promjenu: stereo pretvori u mono, preuzorkuje je
na `FS`, odreže tišinu na početku, skrati je na `MAX_SNIMKA_S` i normira vrh na 0,9.

**Licenca.** Mapa `moj_zvuk/` nije u gitu, pa vlastite snimke ostaju samo na vašem računalu.
U repozitorij (mapa za zapise uz bilježnicu, s izvorom i licencom u `LICENSE-sadrzaj.md`) snimka
smije samo ako ima licencu koja dopušta objavu (CC0, CC BY 4.0 ili CC BY-SA 4.0) i ako se na
njoj ne čuju drugi ljudi bez njihova pristanka.

## 1. Dva digitalna filtra

Zapis su uzorci $x[n]$ uz frekvenciju uzorkovanja $f_s$, s periodom $T_s = 1/f_s$. Frekvencija
$f$ u hercima je na jediničnoj kružnici točka $e^{j\varOmega}$ uz $\varOmega = 2\pi f/f_s$
rad/uzorak: $f = 0$ je $z = 1$, a Nyquistova frekvencija $f_s/2$ je $\varOmega = \pi$, tj.
$z = -1$. Modul frekvencijskog odziva $|H(e^{j\varOmega})|$ zato crtamo u hercima od $0$ do
$f_s/2$, uz $\varOmega$ na gornjoj osi.

**Pomični prosjek (FIR).** Prosjek zadnjih $M$ uzoraka (izraz za pomični prosjek u poglavlju 15)

$$y[n] = \frac1M\sum_{k=0}^{M-1} x[n-k], \qquad H(z) = \frac1M\sum_{k=0}^{M-1} z^{-k}
= \frac{1}{M}\,\frac{z^M - 1}{z^{M-1}(z - 1)}$$

ima $M-1$ nula na jediničnoj kružnici, $e^{j2\pi k/M}$ uz $k = 1, \dots, M-1$, i $(M-1)$-struki
pol u nuli (nula i pol u $z = 1$ se krate). Nula s kutom $2\pi k/M$ je frekvencija
$k f_s/M$: sinusoidu te frekvencije prosjek potpuno uklanja. Modul je
$|H(e^{j\varOmega})| = \left|\frac{\sin(M\varOmega/2)}{M\sin(\varOmega/2)}\right|$, a kašnjenje
$(M-1)/2$ uzoraka (linearna faza).

**IIR filtar prvog reda iz RC kruga.** Uzor je RC krug $G(s) = \dfrac{1}{\tau_f s + 1}$ s
graničnom frekvencijom $f_c = \dfrac{1}{2\pi\tau_f}$, uz $|G(j2\pi f)| = 1/\sqrt{1 + (f/f_c)^2}$.
Tri metode iz poglavlja 15 daju tri diskretna modela prvog reda:

| metoda | zamjena | $H(z)$ | pol | nula |
|---|---|---|---|---|
| Euler | $s \to \dfrac{z-1}{T_s}$ | $\dfrac{T_s/\tau_f}{z - (1 - T_s/\tau_f)}$ | $1 - T_s/\tau_f$ | – |
| ZOH | odziv na skok u $nT_s$ | $\dfrac{1 - a}{z - a}$ | $a = e^{-T_s/\tau_f}$ | – |
| Tustin | $s \to \dfrac{2}{T_s}\,\dfrac{z-1}{z+1}$ | $\dfrac{T_s}{2\tau_f + T_s}\,\dfrac{z+1}{z - \frac{2\tau_f - T_s}{2\tau_f + T_s}}$ | $\dfrac{2\tau_f - T_s}{2\tau_f + T_s}$ | $z = -1$ |

Svi modeli imaju $|H| = 1$ na $f = 0$ ($z = 1$ odgovara $s = 0$), kao RC krug. ZOH preslikava
pol RC kruga $-1/\tau_f$ egzaktno, kao uzorkovanje, u $e^{-T_s/\tau_f}$. Euler je dobra
aproksimacija samo uz $T_s \ll \tau_f$; prema teoremu o polovima Eulerova modela RC krug daje
stabilan model samo uz $T_s < 2\tau_f$, pa uz zadani `FS` klizač $f_c$ ostaje ispod te
granice. Tustinova
metoda čuva stabilnost za svaki $T_s$, a nula u $z = -1$ daje $|H| = 0$ na $f_s/2$. Njezina
frekvencijska os je **savijena**: $H_T(e^{j\varOmega}) = G(j\omega)$ uz $\omega = \frac{2}{T_s}\tan\frac{\varOmega}{2}$,
pa lom ($-3$ dB) Tustinova modela pada na $f = \frac{f_s}{\pi}\arctan\frac{\pi f_c}{f_s}$, niže od
$f_c$. Kad je $f_c$ mala prema $f_s$, razlika je zanemariva.

Ćelija ispod sadrži funkcije modela, frekvencijskog odziva, nula i polova, zadanog zapisa i
ispisa.

In [ ]:
from scipy import signal


def broj(v, decimala=0):
    # broj s decimalnim zarezom i pravim minusom (−3,9); −0 piše se 0
    tekst = f"{v:.{decimala}f}"
    if float(tekst) == 0:
        tekst = tekst.lstrip("-")
    return tekst.replace(".", ",").replace("-", "−")


def tau_iz_fc(f_c):
    # vremenska konstanta RC kruga s graničnom frekvencijom f_c: τ_f = 1/(2π f_c)
    return 1 / (2 * np.pi * f_c)


def fir(M):
    # pomični prosjek duljine M: (b, a) u potencijama od z^−1, kao za signal.lfilter
    return np.ones(M) / M, np.array([1.0])


def euler(tau, Ts):
    # Euler unaprijed, s → (z − 1)/T_s: H(z) = (T_s/τ)/(z − (1 − T_s/τ))
    r = Ts / tau
    return np.array([0.0, r]), np.array([1.0, -(1 - r)])


def zoh(tau, Ts):
    # zadržavanje nultog reda: H(z) = (1 − a)/(z − a), a = e^{−T_s/τ}
    a = np.exp(-Ts / tau)
    return np.array([0.0, 1 - a]), np.array([1.0, -a])


def tustin(tau, Ts):
    # Tustin, s → (2/T_s)(z − 1)/(z + 1): H(z) = c(z + 1)/(z − a),
    # c = T_s/(2τ + T_s), a = (2τ − T_s)/(2τ + T_s)
    c = Ts / (2 * tau + Ts)
    return np.array([c, c]), np.array([1.0, -(2 * tau - Ts) / (2 * tau + Ts)])


def model(filtar, f_c, M):
    # (b, a) filtra iz FILTRI uz graničnu frekvenciju f_c RC kruga i duljinu prosjeka M
    if filtar == "pomični prosjek":
        return fir(M)
    metode = {"Euler": euler, "ZOH": zoh, "Tustin": tustin}
    if filtar not in metode:
        raise ValueError(f"nepoznat filtar „{filtar}”; izaberite jedan od {FILTRI}")
    return metode[filtar](tau_iz_fc(f_c), 1 / FS)


def predizoblicen(f_c):
    # Tustin s predizobličenjem: u RC krug se upiše f_c' = (f_s/π)·tan(π f_c/f_s), pa lom
    # Tustinova modela padne točno na f_c
    return tustin(tau_iz_fc(FS / np.pi * np.tan(np.pi * f_c / FS)), 1 / FS)


def odziv(b, a, f):
    # H(e^{jΩ}) uz Ω = 2πf/f_s (f u Hz, broj ili niz)
    Om = 2 * np.pi * np.asarray(f, dtype=float) / FS
    e = np.exp(-1j * np.multiply.outer(Om, np.arange(max(len(b), len(a)))))
    return (e[..., :len(b)] @ b) / (e[..., :len(a)] @ a)


def rc_odziv(f_c, f):
    # RC krug: G(j2πf) = 1/(1 + j f/f_c)
    return 1 / (1 + 1j * np.asarray(f, dtype=float) / f_c)


def db(H):
    # modul u decibelima, 20 log10 |H| (najmanje −120 dB)
    return 20 * np.log10(np.maximum(np.abs(H), 1e-6))


def nule_polovi(b, a):
    # nule i polovi H(z): brojnik i nazivnik u potencijama od z^−1 pomnoženi s z^n
    n = max(len(b), len(a))
    b = np.concatenate([b, np.zeros(n - len(b))])
    a = np.concatenate([a, np.zeros(n - len(a))])
    return np.roots(b), np.roots(a)


def lom_tustin(f_c):
    # lom (−3 dB) Tustinova modela: Ω = 2 arctan(ω_c T_s/2), tj. f = (f_s/π) arctan(π f_c/f_s)
    return FS / np.pi * np.arctan(np.pi * f_c / FS)


def filtriraj(x, b, a):
    # izlaz filtra iz nultog stanja (jednadžba diferencija)
    return signal.lfilter(b, a, x)


def spektar_db(x):
    # spektralna gustoća snage (Welchov postupak, Hannov prozor, odsječci od 1024 uzorka) u dB
    f, P = signal.welch(x, fs=FS, nperseg=min(1024, len(x)))
    return f, 10 * np.log10(np.maximum(P, 1e-14))


def sinteza():
    # zadani zapis uz FS: melodija tonova s harmonicima (meki početak i kraj svakog tona),
    # šum i stalni pisak
    rng = np.random.default_rng(SJEME)
    t = np.arange(int(TRAJANJE_TONA * FS)) / FS
    ovojnica = np.minimum(1, np.minimum(t, t[-1] - t) / 0.03)       # 30 ms prijelaza
    tonovi = []
    for f0 in MELODIJA:
        ton = sum(C * np.sin(2 * np.pi * k * f0 * t) for k, C in enumerate(HARMONICI, start=1))
        tonovi.append(ovojnica * ton)
    s = np.concatenate(tonovi)
    t = np.arange(len(s)) / FS
    return s + C_PISAK * np.sin(2 * np.pi * F_PISAK * t) + SUM * rng.standard_normal(len(s))


def _razlomak(b, a):
    # tekst prijenosne funkcije prvog reda: b1/(z − p) ili c·(z + 1)/(z − p)
    pol = -a[1]
    nazivnik = f"z {'−' if pol >= 0 else '+'} {broj(abs(pol), 4)}"
    if b[0] == 0:
        return f"{broj(b[1], 4)}/({nazivnik})"
    return f"{broj(b[0], 4)}·(z + 1)/({nazivnik})"


def _modul(H):
    return "nula" if abs(H) < 1e-5 else f"{broj(db(H), 1)} dB"


def opis(f_c, M, filtar):
    # retci ispisa za graničnu frekvenciju f_c, duljinu prosjeka M i filtar koji se sluša
    tau, Ts = tau_iz_fc(f_c), 1 / FS
    retci = [f"f_s = {FS} Hz, T_s = 1/f_s = {broj(1e6 * Ts, 1)} µs; RC krug: f_c = {f_c} Hz, "
             f"τ_f = 1/(2π f_c) = {broj(1e6 * tau, 1)} µs, T_s/τ_f = {broj(Ts / tau, 4)}"]
    if M == 1:
        retci.append("pomični prosjek, M = 1: filtar propušta ulaz nepromijenjen")
    else:
        retci.append(f"pomični prosjek, M = {M}: nule na jediničnoj kružnici (njih M − 1 = "
                     f"{M - 1}) na višekratnicima od f_s/M = {broj(FS / M, 1)} Hz; kašnjenje "
                     f"(M − 1)/2 = {broj((M - 1) / 2, 1)} uzorka = "
                     f"{broj(1000 * (M - 1) / 2 / FS, 3)} ms")
    for ime in ("Euler", "ZOH", "Tustin"):
        b, a = model(ime, f_c, M)
        redak = f"{ime + ':':<8}H(z) = {_razlomak(b, a)}, pol {broj(-a[1], 4)}"
        if ime == "Tustin":
            pomak = f_c - lom_tustin(f_c)
            redak += (", nula −1; lom (−3 dB) praktički na f_c" if pomak < 0.5 else
                      f", nula −1; lom (−3 dB) na {broj(lom_tustin(f_c))} Hz, {broj(pomak)} Hz "
                      "ispod f_c (savijanje frekvencije)")
        retci.append(redak)
    moduli = [f"RC krug {_modul(rc_odziv(f_c, F_PISAK))}"]
    moduli += [f"{ime} {_modul(odziv(*model(ime, f_c, M), F_PISAK))}" for ime in FILTRI]
    retci.append(f"|H| na {F_PISAK} Hz (pisak zadanog zapisa): " + ", ".join(moduli))
    retci.append(f"sluša se izlaz filtra: {filtar}")
    return retci


def opis_snimke(s, ime, fs_izvorno=None):
    # retci ispisa za zapis (s je već uz FS; fs_izvorno je frekvencija uzorkovanja datoteke);
    # javlja grešku za prekratak zapis
    if len(s) < FS // 4:
        raise ValueError(f"{ime}: zapis je prekratak; treba barem {broj(0.25, 2)} s zvuka.")
    if fs_izvorno is None or fs_izvorno == FS:
        uzorkovanje = f"f_s = {FS} Hz"
    else:
        uzorkovanje = f"izvorno f_s = {fs_izvorno} Hz, filtrira se uz f_s = {FS} Hz"
    return [f"{ime}: {uzorkovanje}, {len(s)} uzoraka, trajanje {broj(len(s) / FS, 2)} s",
            f"T_s = 1/f_s = {broj(1e6 / FS, 1)} µs; najviša frekvencija f_s/2 = "
            f"{broj(FS / 2)} Hz (Ω = π, z = −1)"]


def tablica_knjige():
    # funkcije bilježnice za primjere iz knjige: pomični prosjek i tri modela RC kruga
    b, a = fir(KNJIGA_FIR_M)
    Om = 2 * np.pi * KNJIGA_FIR_F / KNJIGA_FIR_FS
    H = abs(np.sum(b * np.exp(-1j * Om * np.arange(len(b)))))
    retci = [f"pomični prosjek, f_s = {KNJIGA_FIR_FS} Hz, M = {KNJIGA_FIR_M}: |H| na "
             f"{KNJIGA_FIR_F} Hz = {broj(H, 3)} (knjiga {broj(KNJIGA_FIR_H, 3)})",
             f"RC krug τ = {broj(KNJIGA_RC_TAU, 1)} s uz T_s = {broj(KNJIGA_RC_TS, 1)} s:"]
    for ime, fun, knjiga in (("Euler", euler, KNJIGA_POL_EULER), ("ZOH", zoh, KNJIGA_POL_ZOH),
                             ("Tustin", tustin, KNJIGA_POL_TUSTIN)):
        b, a = fun(KNJIGA_RC_TAU, KNJIGA_RC_TS)
        redak = f"  {ime + ':':<8}pol {broj(-a[1], 4)} (knjiga {broj(knjiga, 4)})"
        if ime == "Tustin":
            redak += (f", koeficijent uz (z + 1) {broj(b[0], 4)} "
                      f"(knjiga {broj(KNJIGA_TUSTIN_B, 4)})")
        retci.append(redak)
    return retci

In [ ]:
# Učitavanje vlastitog zapisa (WAV), kao u bilježnicama P08 i P13: 8, 16, 24 i 32 bita, stereo u
# mono, preuzorkovanje na FS; za datoteku koja ne postoji, nije WAV ili je tiha javlja grešku
# s uputom. Svaku promjenu snimke ispiše.
PRAG_TISINE = 0.03       # tišina na početku: ispod 3 % vršne vrijednosti (−30 dB)
UPUTA_WAV = ("Bilježnica čita samo WAV. Pretvorite zapis u WAV, npr. u Audacityju: "
             "File → Open, zatim File → Export Audio… (vrsta WAV, Signed 16-bit PCM).")


def zarez2(v):
    return f"{v:.2f}".replace(".", ",")


def nadji(ime, mapa=MOJ_ZVUK):
    # putanja snimke: ime s mapom (npr. "moj_zvuk/zvizduk.wav") ili samo ime u mapi
    put = Path(ime).expanduser()
    kandidati = [put] if put.parent != Path(".") else list(dict.fromkeys(
        [Path(mapa) / put, MOJ_ZVUK / put, put]))
    for k in kandidati:
        if k.is_file():
            return k
    trazeno = ", ".join(k.as_posix() for k in kandidati)
    raise FileNotFoundError(f"Nema datoteke „{ime}” (traženo: {trazeno}). Putanja je "
                            "relativna prema mapi bilježnice, npr. \"moj_zvuk/zvizduk.wav\".")


def preuzorkuj(s, fs_zapisa, fs, ime=""):
    # preuzorkovanje s fs_zapisa na fs (polifazni filtar, scipy.signal.resample_poly)
    if fs_zapisa == fs:
        return s
    d = gcd(fs, fs_zapisa)
    print(f"{ime}: preuzorkovano s {fs_zapisa} Hz na {fs} Hz")
    return resample_poly(s, fs // d, fs_zapisa // d)


def ucitaj(ime, fs=None, mapa=MOJ_ZVUK):
    # WAV -> (frekvencija uzorkovanja, mono signal u rasponu −1 … 1); uz zadani fs zapis s
    # drugom frekvencijom uzorkovanja preuzorkuje se na fs
    put = nadji(ime, mapa)
    if put.suffix.lower() != ".wav":
        raise ValueError(f"{put.name}: nije WAV datoteka. {UPUTA_WAV}")
    try:
        with warnings.catch_warnings():        # dodatni podaci u WAV-u s mobitela
            warnings.simplefilter("ignore", wavfile.WavFileWarning)
            fs_zapisa, s = wavfile.read(put)
    except ValueError as e:                    # npr. MP3 preimenovan u .wav, komprimirani WAV
        raise ValueError(f"{put.name}: ne mogu pročitati kao WAV ({e}). {UPUTA_WAV}") from None
    if s.dtype == np.uint8:                    # 8-bitni WAV: 0 … 255, tišina = 128
        s = (s.astype(float) - 128) / 128
    elif np.issubdtype(s.dtype, np.integer):   # 16, 24 (učitan kao 32) i 32 bita
        s = s / np.iinfo(s.dtype).max
    s = np.asarray(s, dtype=float)
    if s.ndim == 2:                            # stereo -> mono
        s = s.mean(axis=1)
    if s.size == 0 or not np.max(np.abs(s)) > 1e-3:
        raise ValueError(f"{put.name}: zapis je prazan ili tih (vrh ispod −60 dB). "
                         "Provjerite snimku, npr. u Audacityju.")
    if fs is not None:
        s = preuzorkuj(s, fs_zapisa, fs, put.name)
        fs_zapisa = fs
    return fs_zapisa, s


def pripremi(s, fs):
    # vlastita snimka: bez tišine na početku, najviše MAX_SNIMKA_S sekundi, vrh 0,9
    pocetak = int(np.flatnonzero(np.abs(s) > PRAG_TISINE * np.max(np.abs(s)))[0])
    if pocetak > int(0.05 * fs):
        print(f"snimka: odrezano {zarez2(pocetak / fs)} s tišine na početku")
        s = s[pocetak:]
    if len(s) > MAX_SNIMKA_S * fs:
        print(f"snimka: traje {zarez2(len(s) / fs)} s; uzeto prvih {MAX_SNIMKA_S} s "
              "(MAX_SNIMKA_S u postavkama)")
        s = s[:int(MAX_SNIMKA_S * fs)]
    print("snimka: vrh normiran na 0,9")
    return 0.9 * s / np.max(np.abs(s))

## 2. Zapis: melodija, šum i pisak

Ćelija ispod učita vlastiti zapis ili izračuna zadani i ispiše njegove podatke. Zadani zapis je
melodija od deset tonova (osnovni tonovi od 262 do 392 Hz, svaki s tri viša harmonika, poglavlje
9), uz **šum** sa svim frekvencijama do $f_s/2$ (šuštanje) i stalni **pisak** na `F_PISAK`. Oba
niskopropusna filtra propuštaju melodiju i guše šum na visokim frekvencijama. Pisak leži iznad
melodije, pa ga filtar s nulom točno na njegovoj frekvenciji može potpuno ukloniti.

Lijevo je odsječak zapisa od 20 ms, a desno spektralna gustoća snage u decibelima (Welchov
postupak, prosjek spektara odsječaka s Hannovim prozorom, poglavlje 13): vrhovi melodije, ravno
dno šuma i usamljeni vrh piska.

In [ ]:
if SNIMKA:
    fs_izvorno, zapis = ucitaj(SNIMKA)
    ime_zapisa = Path(SNIMKA).name
    zapis = pripremi(preuzorkuj(zapis, fs_izvorno, FS, ime_zapisa), FS)
else:
    fs_izvorno = FS
    zapis, ime_zapisa = sinteza(), "zadani sintetički zapis"
for redak in opis_snimke(zapis, ime_zapisa, fs_izvorno):
    print(redak)
f_ulaz, S_ulaz = spektar_db(zapis)               # spektar ulaza, za usporedbu s izlazom

fig, (ax_t, ax_s) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.5]})
n20 = min(len(zapis), int(0.02 * FS))
pocetak = min(len(zapis) - n20, int(0.15 * FS))
ax_t.plot(1000 * np.arange(n20) / FS, zapis[pocetak:pocetak + n20], color=TAMNA, lw=0.8)
ax_t.set_xlabel("vrijeme od početka odsječka / ms")
ax_t.set_ylabel("$x[n]$")
ax_t.set_title(f"{ime_zapisa}: odsječak od 20 ms", loc="left", fontsize=10)
ax_s.plot(f_ulaz, S_ulaz, color=TAMNA, lw=1)
if not SNIMKA:
    ax_s.axvline(F_PISAK, color=BOJA, ls=":", lw=1.4, label="pisak")
    ax_s.legend(loc="upper right", frameon=True, framealpha=0.9, fontsize=9)
ax_s.set_xlim(0, FS / 2)
ax_s.set_xlabel("frekvencija $f$ / Hz")
ax_s.set_ylabel("dB")
ax_s.set_title("spektralna gustoća snage", loc="left", fontsize=10)
for a in (ax_t, ax_s):
    a.xaxis.set_major_formatter(zarez)
    a.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()
print("Ulaz (prije filtra):")
display(Audio(np.clip(zapis, -1, 1), rate=FS, normalize=False))

## 3. Provjera prema knjizi

Iste funkcije računaju i primjere iz poglavlja 15: pomični prosjek mikrokontrolera (primjer
pomičnog prosjeka kao FIR sustava, modul na smetnji između nula) i polove triju modela RC kruga
iz primjera Eulerove metode, ZOH i Tustinove metode. Tamo je $T_s/\tau$ mnogo veći nego za
zvuk, a polovi su ipak isti izrazi kao u tablici iznad.

In [ ]:
for redak in tablica_knjige():
    print(redak)
b, a = fir(KNJIGA_FIR_M)
Om = 2 * np.pi * KNJIGA_FIR_F / KNJIGA_FIR_FS
assert round(abs(np.sum(b * np.exp(-1j * Om * np.arange(len(b))))), 3) == KNJIGA_FIR_H
for fun, knjiga in ((euler, KNJIGA_POL_EULER), (zoh, KNJIGA_POL_ZOH), (tustin, KNJIGA_POL_TUSTIN)):
    b, a = fun(KNJIGA_RC_TAU, KNJIGA_RC_TS)
    assert round(-a[1], 4) == knjiga
assert round(tustin(KNJIGA_RC_TAU, KNJIGA_RC_TS)[0][0], 4) == KNJIGA_TUSTIN_B
print("Bilježnica i knjiga daju iste brojke.")

## 4. Klizači: granična frekvencija $f_c$, duljina prosjeka $M$ i filtar koji slušate

Klizač **f_c** bira graničnu frekvenciju RC kruga ($\tau_f = 1/(2\pi f_c)$) za sva tri IIR
modela, klizač **M** duljinu pomičnog prosjeka, a gumbi **filtar** čiji izlaz slušate i crtate
dolje.

- **Gore lijevo:** $|H(e^{j\varOmega})|$ u decibelima od $0$ do $f_s/2$ za pomični prosjek i tri
  modela, uz široku sivu krivulju RC kruga $|G|$. Gornja os je $\varOmega$ u rad/uzorak.
  Točkasta okomica je $f_c$, a crtkana pisak zadanog zapisa.
- **Gore desno:** $z$-ravnina s jediničnom kružnicom: nule (kružići) i polovi (križići i
  ostale oznake) svih četiriju filtara.
- **Dolje:** spektralna gustoća snage ulaza (sivo) i izlaza odabranog filtra.

Ispod grafa ispis navodi $T_s/\tau_f$, nule pomičnog prosjeka, $H(z)$ i pol svakog modela, lom
Tustinova modela i modul svih filtara na frekvenciji piska. Zatim se mogu poslušati ulaz i
izlaz odabranog filtra.

Ako se klizači ne prikažu, pozovite funkciju izravno, npr. `pokus(700, 6, "Euler")`.

In [ ]:
f_os = np.linspace(0, FS / 2, 801)
OZNAKE_POLA = {"Euler": dict(marker="s", mfc="white", mec=SIVA, ms=8, mew=1.6),
               "ZOH": dict(marker="D", mfc=SIVA, mec=SIVA, ms=6),
               "Tustin": dict(marker="^", mfc=BOJA, mec=BOJA, ms=8)}


def pokus(f_c=F_C_POCETNA, M=M_POCETNI, filtar=FILTAR_POCETNI):
    modeli = {ime: model(ime, f_c, M) for ime in FILTRI}
    fig = plt.figure(figsize=(12, 8.2), layout="constrained")
    mreza = fig.add_gridspec(2, 2, width_ratios=[1.45, 1])
    ax_h = fig.add_subplot(mreza[0, 0])
    ax_z = fig.add_subplot(mreza[0, 1])
    ax_s = fig.add_subplot(mreza[1, :])

    # modul frekvencijskog odziva u decibelima
    ax_h.plot(f_os, db(rc_odziv(f_c, f_os)), color=SIVA, lw=5, alpha=0.3, label="RC krug $|G|$")
    for ime in FILTRI:
        oznaka = f"pomični prosjek, $M$ = {M}" if ime == "pomični prosjek" else ime
        ax_h.plot(f_os, db(odziv(*modeli[ime], f_os)), label=oznaka, **STIL[ime])
    ax_h.axvline(f_c, color=TAMNA, ls=":", lw=1)
    if not SNIMKA:
        ax_h.axvline(F_PISAK, color=BOJA, ls="--", lw=0.8)
    ax_h.set_xlim(0, FS / 2)
    ax_h.set_ylim(-60, 15)
    ax_h.set_xlabel("frekvencija $f$ / Hz")
    ax_h.set_ylabel("$20\\log_{10}|H|$ / dB")
    ax_h.set_title(f"$f_c$ = {f_c} Hz, $M$ = {M}", loc="left", fontsize=10)
    om = ax_h.secondary_xaxis("top", functions=(lambda f: 2 * np.pi * f / FS,
                                                lambda w: w * FS / (2 * np.pi)))
    om.set_xlabel("Ω / rad/uzorak", fontsize=9)
    om.xaxis.set_major_formatter(zarez)
    ax_h.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=5, frameon=False,
                fontsize=8)

    # nule i polovi u z-ravnini
    kut = np.linspace(0, 2 * np.pi, 400)
    ax_z.plot(np.cos(kut), np.sin(kut), color=SIVA, lw=0.8)
    nule, polovi = nule_polovi(*modeli["pomični prosjek"])
    ax_z.plot(nule.real, nule.imag, "o", mfc="white", mec=TAMNA, ms=7, mew=1.4,
              label="nule prosjeka")
    if len(polovi):
        ax_z.plot([0], [0], "x", color=TAMNA, ms=9, mew=1.8, label=f"pol prosjeka ({M - 1}×)")
    for ime in ("Euler", "ZOH", "Tustin"):
        nule, polovi = nule_polovi(*modeli[ime])
        ax_z.plot(polovi.real, polovi.imag, ls="none", label=f"pol: {ime}", **OZNAKE_POLA[ime])
    ax_z.plot([-1], [0], "o", mfc=BOJA, mec=BOJA, ms=6, label="nula: Tustin")
    ax_z.set_xlim(-1.25, 2.9)                    # desno od kružnice mjesto za legendu
    ax_z.set_ylim(-1.25, 1.25)
    ax_z.set_aspect("equal")
    ax_z.set_xticks([-1, -0.5, 0, 0.5, 1])
    ax_z.set_xlabel("Re $z$")
    ax_z.set_ylabel("Im $z$")
    ax_z.set_title("$z$-ravnina", loc="left", fontsize=10)
    ax_z.legend(loc="center right", frameon=False, fontsize=8)

    # spektar ulaza i izlaza odabranog filtra
    izlaz = filtriraj(zapis, *modeli[filtar])
    ax_s.plot(f_ulaz, S_ulaz, color=SIVA, lw=1, alpha=0.8, label="ulaz")
    ax_s.plot(*spektar_db(izlaz), color=BOJA, lw=1.2, label=f"izlaz: {filtar}")
    ax_s.set_xlim(0, FS / 2)
    ax_s.set_ylim(S_ulaz.max() - 90, S_ulaz.max() + 8)
    ax_s.set_xlabel("frekvencija $f$ / Hz")
    ax_s.set_ylabel("dB")
    ax_s.set_title("spektralna gustoća snage ulaza i izlaza", loc="left", fontsize=10)
    ax_s.legend(loc="upper right", frameon=True, framealpha=0.9, fontsize=9)
    for a in (ax_h, ax_z, ax_s):
        a.xaxis.set_major_formatter(zarez)
        a.yaxis.set_major_formatter(zarez)
    plt.show()
    for redak in opis(f_c, M, filtar):
        print(redak)
    print("Ulaz (prije filtra):")
    display(Audio(np.clip(zapis, -1, 1), rate=FS, normalize=False))
    print("Izlaz odabranog filtra (poslije):")
    display(Audio(np.clip(izlaz, -1, 1), rate=FS, normalize=False))


klizac = widgets.interactive(
    pokus,
    f_c=widgets.SelectionSlider(options=F_C_IZBOR, value=F_C_POCETNA, description="f_c / Hz",
                                continuous_update=False, layout=widgets.Layout(width="520px")),
    M=widgets.IntSlider(value=M_POCETNI, min=1, max=M_NAJVECI, description="M",
                        continuous_update=False, layout=widgets.Layout(width="520px")),
    filtar=widgets.ToggleButtons(options=FILTRI, value=FILTAR_POCETNI, description="filtar"),
)
display(klizac)

## 5. Tri metode uz nisku i visoku graničnu frekvenciju

Slika ispod uspoređuje RC krug i tri modela uz nisku i najvišu $f_c$ s klizača, bez zvuka.
Desno je dodan i Tustinov model s **predizobličenjem**: u RC krug se upiše granična frekvencija
$f_c' = \frac{f_s}{\pi}\tan\frac{\pi f_c}{f_s}$, pa nakon savijanja lom padne točno na $f_c$.

- **Niska $f_c$** ($T_s \ll \tau_f$): polovi svih triju modela su blizu $z = 1$ i gotovo
  jednaki, a moduli prate RC krug do nekoliko kiloherca. Razilaze se tek prema $f_s/2$: Tustin
  zbog nule u $z = -1$ pada na nulu, a Euler i ZOH ondje ne padaju na nulu (na slici ostaju
  oko $-20$ dB).
- **Visoka $f_c$** ($T_s$ nije mnogo manji od $\tau_f$): modeli se razilaze. Eulerov pol
  $1 - T_s/\tau_f$ odmakne se daleko od točnog $e^{-T_s/\tau_f}$, pa Eulerov modul blizu $f_s/2$
  raste umjesto da pada. ZOH ima točan pol, ali na $f_s/2$ ne pada na nulu. Tustin ima lom niže
  od $f_c$ (savijanje frekvencije), a predizobličenje ga vrati na $f_c$.

Zato se za digitalni filtar koji zamjenjuje analogni obično bira Tustin, uz predizobličenje kad
je važna jedna frekvencija (pravilo o izboru metode diskretizacije u poglavlju 15).

In [ ]:
fig, osi = plt.subplots(1, 2, figsize=(12, 3.9), sharey=True)
for ax, f_c in zip(osi, (F_C_IZBOR[2], F_C_IZBOR[-1])):
    ax.plot(f_os, db(rc_odziv(f_c, f_os)), color=SIVA, lw=5, alpha=0.3, label="RC krug $|G|$")
    for ime in ("Euler", "ZOH", "Tustin"):
        ax.plot(f_os, db(odziv(*model(ime, f_c, 1), f_os)), label=ime, **STIL[ime])
    if f_c == F_C_IZBOR[-1]:
        ax.plot(f_os, db(odziv(*predizoblicen(f_c), f_os)), color=BOJA, ls="-", lw=1,
                label="Tustin, predizobličen")
    ax.axvline(f_c, color=TAMNA, ls=":", lw=1)
    ax.axhline(-3, color=SIVA, ls="-", lw=0.6)
    ax.set_xlim(0, FS / 2)
    ax.set_ylim(-40, 15)
    ax.set_xlabel("frekvencija $f$ / Hz")
    ax.set_title(f"$f_c$ = {f_c} Hz (točkasto), siva vodoravna: −3 dB", loc="left", fontsize=10)
    ax.xaxis.set_major_formatter(zarez)
    ax.yaxis.set_major_formatter(zarez)
osi[0].set_ylabel("$20\\log_{10}|H|$ / dB")
osi[0].legend(loc="lower left", frameon=True, framealpha=0.9, fontsize=8)
osi[1].legend(loc="lower left", frameon=True, framealpha=0.9, fontsize=8)
fig.tight_layout()
plt.show()

## Pokušajte sami

1. Uz zadanu $f_c$ poslušajte izlaz pomičnog prosjeka za nekoliko vrijednosti $M$. Uz koji $M$
   pisak potpuno nestane? Ima li na klizaču još takvih $M$? Gdje su tada nule u $z$-ravnini?
2. Uz $M$ koji uklanja pisak usporedite zvuk pomičnog prosjeka i Tustinova modela uz
   $f_c = 1000$ Hz. Koji filtar bolje čuva melodiju, a koji više guši šum? Objasnite to
   modulima obaju filtara.
3. Pomičite $f_c$ i pratite polove triju modela u $z$-ravnini. Uz koju $f_c$ s klizača Eulerov
   pol prvi put prijeđe na negativnu realnu os? Izračunajte to iz izraza za Eulerov pol.
4. Izračunajte lom Tustinova modela uz $f_c = 4000$ Hz iz izraza za savijanje frekvencije i
   provjerite ga u ispisu. Koju bi $f_c'$ trebalo upisati u RC krug da lom padne na 4000 Hz?
5. Snimite vlastiti zapis (govor ili glazba uz šum ventilatora), spremite ga kao WAV u
   `moj_zvuk/` i upišite u postavke. Uz koju $f_c$ šum najviše oslabi, a govor ostane
   razumljiv?
6. U postavkama zadajte `FS = 32000` i pokrenite sve ćelije. Kako se uz istu $f_c$ pomaknu
   polovi triju modela? Razlikuju li se modeli sada više ili manje?

---
Kod je pod licencom MIT (`LICENSE`), a tekst pod CC BY 4.0 (`LICENSE-sadrzaj.md`).